### Retrieval

**RAG의 5단계**
1. **Document Loader**: 문서를 불러오고
2. **Document Transformer**: 문서를 쪼개고
3. **Embedding**: 텍스트를 숫자로 바꾸고
4. **Vector Store**: 저장소에 넣고
5. **Retrieval**: 검색해서 LLM에 전달합니다.

In [1]:
# %pip install langchain-community pypdf faiss-cpu sentence-transformers

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### Document Loader(문서 불러오기)

In [3]:
# %pip install bs4

In [4]:
from langchain_community.document_loaders import WebBaseLoader # 웹페이지 URL에서 텍스트를 긁어오는 도구

url = "https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EC%A0%95%EC%B1%85%EA%B3%BC_%EC%A7%80%EC%B9%A8"

# 로더 인스턴스 생성
loader = WebBaseLoader(url)

# 해당 URL에 접속하여 HTML 파싱, 텍스트만 추출하여 Document 객체 리스트로 반환.
documents = loader.load()

print(len(documents))
print(documents[0].metadata)

# 본문 내용 확인
print(documents[0].page_content[:500])

c:\Users\Admin\miniconda3\envs\pystudy_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


1
{'source': 'https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EC%A0%95%EC%B1%85%EA%B3%BC_%EC%A7%80%EC%B9%A8', 'title': '위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전', 'language': 'ko'}




위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전



























본문으로 이동







주 메뉴





주 메뉴
사이드바로 이동
숨기기



		둘러보기
	


대문최근 바뀜요즘 화제임의의 문서로





		사용자 모임
	


사랑방사용자 모임관리 요청





		편집 안내
	


소개도움말정책과 지침질문방



















검색











검색






















보이기
















기부

계정 만들기

로그인








개인 도구





기부 계정 만들기 로그인




























목차
사이드바로 이동
숨기기




처음 위치





1
최상위 정책








2
'정책과 지침'이란?








3
준수








4
집행








5
문서 내용








6
정책과 지침은 백과사전의 일부가 아닙니다






In [5]:
from langchain_community.document_loaders import PyPDFLoader # PDF 파일을 로드하여 텍스트로 변환하는 도구

# 로더 인스턴스 생성 (파일 경로 지정)
loader = PyPDFLoader("The_Adventures_of_Tom_Sawyer.pdf")

# 문서 로드 실행 : PDF 각 페이지를 하나의 Document 객체로 변환하여 리스트로 반환
documents = loader.load()

print(len(documents))
print(documents[0].metadata)
print(documents[3].page_content)    # 4번째 페이지(인덱스 3)의 본문 내용

35
{'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}
Pearson Education Limited                                                                            
Edinburgh Gate, Harlow,                                                                               
Essex CM20 2JE, England                                                                              
and Associated Companies throughout the world. 
ISBN 0 582 41923 9 
 
First published 1876                                                                                  
Published by Puffin Books 1950                                                                         
This edition first publis

### Embedding Model(임베딩: 텍스트를 숫자로)

In [6]:
from langchain_openai import OpenAIEmbeddings
import pandas as pd

# 임베딩 모델
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
text="The quick brown for jumps over the lazy dog."
vector = embeddings.embed_query(text) # 하나의 문자열을 벡터로 변환.

print(len(vector))
print(pd.Series(vector).head())

1536
0   -0.003728
1    0.000956
2    0.023028
3   -0.057349
4   -0.011776
dtype: float64


In [9]:
# 문서 내용만 추출
docs = [document.page_content for document in documents]
print(len(docs))

# embed_document() : 문자열 리스트를 받아서, 각가을 벡터로 변환한 뒤 '벡터 리스트'를 반환
vects = embeddings.embed_documents(docs)

print(len(vects))
print(len(vects[0]))
pd.DataFrame(vects)

35
35
1536


,0,1,2,3,4,5,6,7,8,9,...,1526,1527,1528,1529,1530,1531,1532,1533,1534,1535
0,0.015368,-0.034811,-0.009329,0.014481,0.007343,0.014409,-0.052248,0.049236,-0.013593,0.015107,...,-0.008608,0.020671,0.002576,-0.002818,-0.021685,0.024887,0.025030,-0.013593,0.017691,0.019022
1,0.021994,-0.019466,0.014100,-0.002712,0.000633,-0.046833,-0.046287,0.089815,-0.024020,-0.008986,...,0.010896,0.003247,0.030542,-0.008720,-0.015429,0.053872,-0.006594,-0.019063,0.023474,-0.003295
2,-0.011807,-0.009602,0.013972,-0.021397,-0.017350,0.005830,-0.002324,0.046666,-0.026301,-0.010646,...,-0.008962,-0.025526,-0.004490,0.011479,-0.050302,0.033152,0.011408,-0.005062,0.044319,-0.007091
3,0.020604,-0.024787,0.009854,-0.010347,-0.007561,-0.001778,-0.006314,0.012777,-0.050094,-0.016369,...,0.017785,-0.027229,0.007379,-0.018058,-0.047807,0.047677,-0.022163,-0.008724,0.034842,0.009074
4,0.006802,-0.015708,0.023817,-0.000497,-0.017493,-0.030473,0.023486,0.009894,-0.029300,-0.013094,...,-0.041107,-0.024570,-0.022670,0.007408,-0.048833,0.042535,0.010691,-0.001524,0.031366,0.002701
5,0.018520,0.015934,-0.040698,0.035223,0.023566,-0.031564,0.015606,0.020437,-0.047031,0.021068,...,-0.014306,-0.028663,-0.018494,0.009172,-0.024828,0.029193,-0.000720,0.014041,0.023415,-0.016413
6,0.030211,0.047995,-0.024080,-0.001274,0.027674,0.006466,0.030587,-0.017983,-0.040008,-0.009033,...,0.034158,0.037799,-0.021073,-0.015810,-0.008305,0.002179,0.047807,-0.023011,0.033077,-0.025677
7,-0.005991,0.019755,-0.022522,0.019743,0.012449,-0.041662,0.022770,-0.013170,-0.014494,-0.003703,...,-0.017485,-0.034947,-0.031992,0.001977,-0.036248,0.023881,0.006632,0.001101,0.026813,-0.010469
8,-0.012401,-0.000124,-0.045416,0.002174,-0.014618,-0.043433,0.025619,-0.014928,-0.015708,-0.011695,...,-0.027403,-0.014222,-0.040039,-0.010437,-0.024194,0.036199,-0.005380,-0.004962,0.021209,0.009285
9,-0.018840,0.050801,-0.045571,0.008173,0.010839,0.004100,0.041470,0.045238,-0.009331,-0.084979,...,-0.003671,0.012601,-0.030454,-0.012324,0.020192,0.007486,0.039653,-0.032538,0.008539,0.002471


### vector store(FAISS)

In [12]:
# FAISS
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# 벡터 저장소 생성
# 이 함수 내부에서 'documents'의 텍스트를 'embeddings'모델로 벡터화하고, FAISS 인덱스를 만들어 저장한다.
vector_store = FAISS.from_documents(documents, embeddings)  # FAISS.from_documents(벡터화 할 documents, embedding에 사용할 모델) / documents는 위에서 PDF 내용을 담은 것임

print(vector_store)

# 유사도 검색
query="Tom Sawyer"
# .similarity_search() : 질문(query)와 가장 유사한(거리가 가까운) 문서를 찾는다
# k = 3 : 가장 유사한 문서 3개를 가져오라는 뜻
retrieved_docs = vector_store.similarity_search(query, k=3)

print(retrieved_docs)

[Document(id='da435961-0f80-4780-b003-28bca41c95bf', metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 4, 'page_label': '5'}, page_content='Introduction \n \n \nOne Saturday afternoon Tom wanted to have an adventure                    \nbecause he didn’t want to think about Injun Joe. He went \nto Huck and said, “I’m going to  look for treasure. Do you \nwant to come with me?” \n \nTom Sawyer loves adventures. He has a lot of adventures \nat home, at school, and with his friends. He has one \nadventure in a cave. But why is he there? What does he \nsee in the cave? And why is he afraid? \n \nMark Twain (1835-1910) is a famous American writer. \nHis name was Samuel Cl

### Retrieval & RAG (검색기 연결 및 질의응답)

In [18]:
%pip install langchain-classic.chains

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchain-classic.chains (from versions: none)
ERROR: No matching distribution found for langchain-classic.chains


In [29]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

# Retrieval 변환
# 벡터 스토어를 LangChain의 표준 'Retrieval'인터페이스로 변환 -> 나중에 Chain이나 Agent에 끼워 쓸 수 있음
retrieval = vector_store.as_retriever()

# 모델 생성
model = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0   # 사실 기반으로 대답하게 하기 위해 온도를 0으로 낮춤
)

system_prompt = (
    "당신은 질문, 답변을 돕는 보조원입니다. "
    "아래 제공된 context를 사용하여 질문에 답하세요. "
    "답을 모르면 모른다고 하되, 답변을 지어내지 마세요. "
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

# 문서 결합 체인 생성 : 역할 -> 검색된 문서들을 하나로 뭉쳐 프롬프트의 {context} 자리에 채워 넣음
combine_docs_chain = create_stuff_documents_chain(model, prompt)

# 리트리벌 체인 생성 : 역할 -> 질문을 받아 검색기로 문서를 찾고, 그 문서들을 결합 체인으로 넘김
rag_chain = create_retrieval_chain(retrieval, combine_docs_chain)

# 질의응답
response1 = rag_chain.invoke({"input" : "마을 무덤에 있던 남자를 누가, 왜 죽였나요?"})

print("답변1 : ", response1['answer'])

response2 = rag_chain.invoke({"input" : "톰소여는 어떤 사람인가요?"})

print("답변2 : ", response2['answer'])

답변1 :  그 남자는 의사였고, 살해자는 Injun Joe였습니다. 무덤가에서 의사와 Injun Joe가 다투자 Injun Joe가 칼로 의사를 죽였고, Muff Potter는 그 살인자가 아니었습니다.
답변2 :  톰 소여는 모험을 사랑하는 활발한 소년이에요.  
- 집이나 학교, 친구들(Huck Finn, Joe Harper)과 함께 다양한 모험을 즐깁니다.  
- 충성스럽고 친구를 돕고 싶어 하는 마음이 강합니다. 예를 들어 Muff Potter를 돕고 싶어하고 음식을 들고 찾아가기도 합니다.  
- 다소 장난꾸러기이고 때로는 잘못을 저지르기도 합니다(예: 책을 찢었다고 말하는 상황처럼 자신의 행동을 인정하기도 합니다).  
- 한편 Graveyard의 사건 이후 Injun Joe를 다시 두려워하기도 하며, 두려움과 용기가 함께 반영되는 복합적인 면모를 보입니다.  

요약하면, 톰은 모험을 즐기는 동시에 친구를 돕고자 하는 정의감과 충성심이 있는 활발한 소년입니다.
